# Lab 07-01: modelli con visione

Lab: `corso/laboratori/lab-07-01-visione`

Esegui le celle in ordine (Runtime > Esegui tutto). Scarica fino a 1 GB e richiede alcuni minuti.


## 1. Preparazione

Scarica i binari ufficiali di llama.cpp (build `b11046`, quella del corso) e il materiale del lab.

In [ ]:
import os, subprocess
V = "b11046"
HOME = os.path.expanduser("~")
REL = f"{HOME}/llama-lab/llama-{V}"
if not os.path.exists(f"{REL}/llama"):
    subprocess.run(
        f"mkdir -p {HOME}/llama-lab && cd {HOME}/llama-lab && "
        f"curl -sLO https://github.com/ggml-org/llama.cpp/releases/download/{V}/llama-{V}-bin-ubuntu-x64.tar.gz && "
        f"tar -xzf llama-{V}-bin-ubuntu-x64.tar.gz", shell=True, check=True)
os.environ["PATH"] = REL + ":" + os.environ["PATH"]
T = min(len(os.sched_getaffinity(0)), 4)                      # thread = core disponibili (oltre i core fisici si rallenta)
os.environ["T"] = str(T)
!llama version

In [ ]:
BASE = "/content" if os.path.isdir("/content") else HOME
if not os.path.exists(f"{BASE}/corso"):
    !git clone -q --depth 1 https://github.com/JoeBosi/llama-cpp-course.git {BASE}/corso
LABS = f"{BASE}/corso/corso/laboratori"
os.environ["LABS"] = LABS

Il modello è SmolVLM-256M (multimodale, circa 280 MB).

## 2. Un'immagine di prova

Due rettangoli colorati, creati con `ffmpeg` (preinstallato su Colab).

In [ ]:
%%bash
mkdir -p ~/lab07 && cd ~/lab07
ffmpeg -nostdin -loglevel error -y -f lavfi \
  -i "color=c=white:s=320x240:d=1,drawbox=x=40:y=40:w=120:h=120:color=red@1:t=fill,drawbox=x=200:y=60:w=80:h=140:color=blue@1:t=fill" \
  -frames:v 1 forme.png
ls -la forme.png

In [ ]:
from IPython.display import Image
Image(os.path.expanduser("~/lab07/forme.png"))

## 3. Via API (lab 07-01)

In [ ]:
import subprocess, time, urllib.request

def avvia_server(args, porta=8080, attesa=300):
    """Avvia `llama serve` in background e aspetta che risponda su /health."""
    log = open("/tmp/llama-server.log", "w")
    p = subprocess.Popen(["llama", "serve", *args.split(), "--host", "127.0.0.1", "--port", str(porta)],
                         stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(attesa):
        try:
            if urllib.request.urlopen(f"http://127.0.0.1:{porta}/health", timeout=2).status == 200:
                return p
        except Exception:
            time.sleep(1)
    raise RuntimeError("il server non risponde: " + open("/tmp/llama-server.log").read()[-1500:])

def ferma_server(p):
    p.terminate(); p.wait()

In [ ]:
p = avvia_server("-hf ggml-org/SmolVLM-256M-Instruct-GGUF:Q8_0 -c 4096 -t %d" % T)

In [ ]:
%%bash
cd ~/lab07
bash $LABS/lab-07-01-visione/chiedi_immagine.sh forme.png "Describe this image." http://127.0.0.1:8080

In [ ]:
ferma_server(p)